In [ ]:
import json
import hashlib
import re
import time
from collections import Counter

DATA = PROJECT_ROOT / "data" / "expanded_20261108"

MODEL = "text-embedding-3-small"
DIM = 768
INDEX = "career_api_chunk_768"

all_rows = json.loads(
    (DATA / "retrieval_chunks.json").read_text(encoding="utf-8")
)

api_rows = [
    dict(row) for row in all_rows
    if row["source_type"] == "career_api"
]

if len(api_rows) != 6798:
    raise ValueError(
        f"예상 API 청크 6,798개와 다릅니다: {len(api_rows)}"
    )

ids = [row["chunk_id"] for row in api_rows]
if len(ids) != len(set(ids)):
    raise ValueError("중복 chunk_id가 있습니다.")

raw_cache = {}

def normalize_space(value):
    return " ".join(str(value).split())

def resolve_source(raw, locator):
    value = raw
    for part in locator.split("."):
        match = re.fullmatch(r"([^\[]+)(?:\[(\d+)\])?", part)
        if match is None:
            raise ValueError(f"해석할 수 없는 원천 위치: {locator}")
        value = value[match.group(1)]
        if match.group(2) is not None:
            value = value[int(match.group(2))]
    return value

for row in api_rows:
    if not row["text"].strip():
        raise ValueError(f"빈 본문: {row['chunk_id']}")

    source_file = row["source_file"]
    if source_file not in raw_cache:
        raw_cache[source_file] = json.loads(
            (DATA / source_file).read_text(encoding="utf-8")
        )

    original = resolve_source(
        raw_cache[source_file], row["source_locator"]
    )
    if normalize_space(original) != normalize_space(row["text"]):
        raise ValueError(f"원천 본문 불일치: {row['chunk_id']}")

    row["job_id"] = f"job:{row['job_code']}"

    # 기존 API 100개 시험 코드와 동일한 형식
    row["embedding_text"] = (
        f"직업: {row['job_name']}\n"
        f"항목: {row['source_locator']}\n"
        f"내용: {row['text']}"
    )
    row["input_hash"] = hashlib.sha256(
        row["embedding_text"].encode("utf-8")
    ).hexdigest()

api_rows.sort(key=lambda row: row["chunk_id"])
api_ids = [row["chunk_id"] for row in api_rows]
api_by_id = {row["chunk_id"]: row for row in api_rows}

def batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]

print("원천 검증 완료")
print("API 청크:", len(api_rows))
print("문서:", len({r["document_id"] for r in api_rows}))
print("직업:", len({r["job_id"] for r in api_rows}))